The Graph API schema in LangGraph defines the blueprint for your graph's State, which acts as the shared memory between all nodes and edges. [1] 
## Core Schema Types
You can define your graph's state using three main Python structures: [1, 2] 

* TypedDict: The most common and recommended way to specify a schema for its balance of performance and simplicity.
* Dataclass: Ideal if you need to provide default values for your state keys.
* Pydantic BaseModel: Used when you require recursive data validation, though it may be slightly less performant than TypedDict or dataclass. [1] 

## Key Characteristics

* Shared Input/Output: By default, every node in the graph uses the same schema for both input and output.
* Custom Schemas: You can explicitly define different input and output schemas if you have many keys and want to restrict what is exposed at each end.
* Reducers: The schema works alongside reducer functions, which dictate how updates from different nodes are merged into the state (e.g., appending to a list of messages vs. overwriting a single value)


LangGraph Schema behavior:

* State as a Union: The graph state is the total union of all channels defined in OverallState, input_schema, and output_schema.
* Write-Access Decoupling: Nodes are not restricted to writing only to the keys defined in their input schema. They can write to any valid channel within the graph’s total state.
* Dynamic Channel Expansion: Nodes can declare and write to new state channels (like a PrivateState) at runtime, provided the schema definition exists in the code.
* Schema as a Filter: The input_schema and output_schema act as input/output gates. They control what data a node receives and what data the final user sees, respectively.
* Internal Scaffolding: You can use intermediate state keys (like foo or bar) to pass data between nodes without cluttering the final graph output.
* Implicit Registration: LangGraph effectively registers new keys on the fly if a node returns a dictionary containing a key that is defined in a reachable TypedDict.
* Persistence Limitation: While dynamic expansion works in memory, keys intended for Checkpointers or LangGraph Cloud should ideally be explicitly declared in the main state for reliable serialization.



In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class InputState(TypedDict):
    user_input: str

class OutputState(TypedDict):
    graph_output: str

class OverallState(TypedDict):
    foo: str
    user_input: str
    graph_output: str

class PrivateState(TypedDict):
    bar: str

def node_1(state: InputState) -> OverallState:
    # Write to OverallState
    return {"foo": state["user_input"] + " name"}

def node_2(state: OverallState) -> PrivateState:
    # Read from OverallState, write to PrivateState
    return {"bar": state["foo"] + " is"}

def node_3(state: PrivateState) -> OutputState:
    # Read from PrivateState, write to OutputState
    return {"graph_output": state["bar"] + " Lance"}

builder = StateGraph(OverallState,input_schema=InputState,output_schema=OutputState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
graph.invoke({"user_input":"My"})
# {'graph_output': 'My name is Lance'}

{'graph_output': 'My name is Lance'}